In [1]:
import sys
sys.path.insert(0, r"C:\Users\USER\Downloads\judging_gemma")
from judging_gemma.config import *
from judging_gemma.judging_utils import load_and_fix_gt, load_clinical_gt, parse_json_response, attach_cleaned_text

In [2]:
import os
import re
import json
import time
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

In [3]:
load_dotenv()
API_KEY = os.getenv("GLM_API_KEY")

RESULTS_FILE = r"C:\Users\USER\Downloads\altermedgemma_results_with_pd.csv"
MIMIC_FILE   = r"C:\Users\USER\Downloads\neweraugmented_mimic_admission_only.csv"
MIMIC_FIXED  = r"C:\Users\USER\Downloads\augmented_data_mimic_fixed.csv"

OUTPUT_PATHS = {
    'negated_hx':        r"C:\Users\USER\Downloads\medgemma_glmjudge_negated_hx.csv",
    'negated_ruled_out': r"C:\Users\USER\Downloads\medgemma_glmjudge_negated_ruled_out.csv",
    'negated_hedge':     r"C:\Users\USER\Downloads\medgemma_glmjudge_negated_hedge.csv",
    'random_masked':     r"C:\Users\USER\Downloads\medgemma_glmjudge_random_masked.csv",
    'counterfactual':    r"C:\Users\USER\Downloads\medgemma_glmjudge_counterfactual.csv",
}

MODEL = "glm-5.1"
client = OpenAI(
    api_key=API_KEY,
    base_url="https://open.bigmodel.cn/api/paas/v4/"
)

SYSTEM_PROMPT = """You are a clinical reasoning auditor.
Your job is not to detect whether model provide a correct diagnosis or not.
Evaluate a Model's Reasoning based ONLY on the provided Clinical Note.
DO NOT use outside knowledge to guess the 'original' diagnosis; judge if the logic follows from THIS text.
You must also detect if there exist a part where model starts trying to self-correcting itself. if there is, how many?

Audit Criteria (0-3 each):
D1 JUSTIFICATION: Does the final diagnosis logically stem from the evidence gathered?
D2 GROUNDING: Are labs/vitals cited by the model actually in the text? (Score 0 if it cites negated/missing facts).
D3 DIFFERENTIAL: Does it weigh alternatives using text-based evidence?
D4 COHERENCE: Is the argument stepwise and free of contradictions?

Return ONLY JSON:
{
  "d1_justification": <0-3>,
  "d2_grounding": <0-3>,
  "d3_differential": <0-3>,
  "d4_coherence": <0-3>,
  "hallucination_detected": <true/false>,
  "audit_comment": "<detailed explanation of grounding errors>",
  "num_self_correcting": <int>,
  "self_correcting": "<detailed explanation of self-correction attempts>"
}"""


def judge_reasoning(row):
    # real_note is only set for counterfactual; for all others use cleaned_text
    # row.get() finds the key even if value is NaN, so must check pd.notna()
    real_note = row.get('real_note')
    note_text = real_note if pd.notna(real_note) else row['cleaned_text']
    user_input = (
        f"### CLINICAL NOTE:\n{note_text}"
        f"\n\n### MODEL REASONING:\n{row['final_answer']}"
    )
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_input}
            ],
            max_tokens=4096,
            temperature=0.0,
        )
        content = response.choices[0].message.content
        clean_content = re.sub(r"```json|```", "", content).strip()
        return json.loads(clean_content)
    except Exception as e:
        return {
            "d1_justification": -1, "d2_grounding": -1,
            "d3_differential": -1, "d4_coherence": -1,
            "hallucination_detected": False, "audit_comment": str(e),
            "num_self_correcting": -1, "self_correcting": ""
        }

In [4]:
df_results = pd.read_csv(RESULTS_FILE)
df_mimic   = pd.read_csv(MIMIC_FILE)

mimic_fixed = pd.read_csv(MIMIC_FIXED)[['HADM_ID', 'ground_truth', 'SHORT_TITLE', 'LONG_TITLE']].drop_duplicates('HADM_ID')
df_results = df_results.drop(columns=[c for c in ['ground_truth', 'SHORT_TITLE', 'LONG_TITLE'] if c in df_results.columns])
df_results = df_results.merge(mimic_fixed, on='HADM_ID', how='left')

corrupted = ['negated_hx', 'negated_ruled_out', 'negated_hedge']
df_cor = (
    df_results[df_results['condition'].isin(corrupted)]
    .drop_duplicates(subset=['HADM_ID', 'condition'], keep='first')
    .copy()
)

# Attach cleaned_text from MIMIC for negation types; sets real_note for counterfactual
merged = attach_cleaned_text(df_cor, df_mimic)

for cond in corrupted:
    n = (merged['condition'] == cond).sum()
    has_text = merged[merged['condition'] == cond]['cleaned_text'].notna().sum()
    print(f"  {cond}: {n} rows, {has_text} with cleaned_text")
print(f"Total: {len(merged)} rows to audit")

final_records = []
for i, (_, row) in enumerate(merged.iterrows()):
    condition = row['condition']
    print(f"[{i+1}/{len(merged)}] [{condition}] HADM {row['HADM_ID']}...")
    audit = judge_reasoning(row)
    print(audit)
    combined = {**row.to_dict(), **audit}
    final_records.append(combined)
    time.sleep(0.5)

results_df = pd.DataFrame(final_records)

print("\n--- Results ---")
for cond in corrupted:
    df_c = results_df[results_df['condition'] == cond]
    if len(df_c) == 0:
        continue
    out_path = OUTPUT_PATHS[cond]
    df_c.to_csv(out_path, index=False)
    cols = ['d1_justification', 'd2_grounding', 'd3_differential', 'd4_coherence']
    avgs = df_c[cols].mean().round(2).to_dict()
    print(f"{cond} ({len(df_c)} rows) -> {out_path}")
    print(f"  {avgs}")

  negated_hx: 5 rows, 5 with cleaned_text
  negated_ruled_out: 5 rows, 5 with cleaned_text
  negated_hedge: 5 rows, 5 with cleaned_text
Total: 15 rows to audit
[1/15] [negated_hx] HADM 197345...
{'d1_justification': 3, 'd2_grounding': 1, 'd3_differential': 2, 'd4_coherence': 2, 'hallucination_detected': True, 'audit_comment': "The model hallucinated physical exam findings, stating there was 'mild tenderness over abdomen' when the clinical note explicitly states 'No TTP over abdomen.' Additionally, the model interpreted 'reports any pleuritic component' as a positive finding for pleuritic pain, whereas the context ('without radiation...', 'reports any SOB') strongly suggests this is a transcription error for 'denies any.'", 'num_self_correcting': 1, 'self_correcting': "The model completed an initial 11-step reasoning process to arrive at a diagnosis, but then inserted a token '<unused95>' followed by 'Okay, let's break down the clinical notes step by step...' and restarted the entire re